# Marmousi Elastic OPT Debug

Small `2DsismoTimeIsoHetero` check using Marmousi-derived `rho`, `vp`, and `vs`. The material variables passed to the equation are `rho`, `lambda`, and `mu` in MKS units.


## 1. Setup


In [ ]:
using Pkg
cd(@__DIR__)
Pkg.activate("../")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end

using JLD2
ParamFile = "../config/testparam.csv"

include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT

include("temporaryHelpers.jl")


## 2. Reduced Elastic Marmousi Material


In [ ]:
marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")

shape = (41, 41)
downsample_step = 8
rho_raw = downsample_center_crop(marmousi.ρ, shape; step=downsample_step)
vp_raw = downsample_center_crop(marmousi.Vpv, shape; step=downsample_step)
vs_raw = downsample_center_crop(marmousi.Vsv, shape; step=downsample_step)

rho, lambda, mu, vp, vs = elastic_lame_from_rho_vp_vs(rho_raw, vp_raw, vs_raw)

# First small linear test: avoid a singular pure-fluid block. Set to 0.0 when testing the free-surface/fluid behavior explicitly.
vs_floor = 500.0
if vs_floor > 0
    vs = max.(vs, vs_floor)
    mu = rho .* vs.^2
    lambda = rho .* vp.^2 .- 2 .* mu
end

dx = 1.0e3 * downsample_step
cfl = 0.25
cfl_info = cfl_diagnostics(vp, (dx, dx, 1.0); cfl_safety=cfl)
dt = cfl_info.suggested_dt_2D
delta = (dx, dx, dt)
models_elastic = [rho, lambda, mu]

sourcePoint = CartesianIndex(cld(size(rho, 1), 2), cld(size(rho, 2), 2))
Nt = 30
store_every = 2
source_f0 = min(cfl_info.suggested_f0, 1 / (20dt))
source_t0 = 4 / source_f0
source_amplitude = 1e8

@show size(rho) extrema(rho) extrema(vp) extrema(vs) extrema(lambda) extrema(mu)
@show delta sourcePoint Nt store_every source_f0 source_t0


## 3. Build 2DsismoTimeIsoHetero OPT


In [ ]:
elasticOPT = build_opt_prepared(
    "2DsismoTimeIsoHetero",
    models_elastic,
    delta;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_elastic_reduced_OPT",
)
preparedElastic = elasticOPT.prepared
elastic_A_report = implicit_matrix_report(preparedElastic)

@show preparedElastic.spaceShape preparedElastic.NpointsSpace preparedElastic.NField preparedElastic.NForceField preparedElastic.timePointsUsedForOneStep
@show size(preparedElastic.A_unknown) nnz(preparedElastic.A_unknown) nnz(preparedElastic.L_known) nnz(preparedElastic.R_force)
@show elastic_A_report


## 4. Source And Linear Propagation


In [ ]:
timeSignal = source_time_samples(Nt, dt, preparedElastic.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)

# Force fields are moment components: M11, M12, M21, M22.
sourceFull = point_source_full(preparedElastic, sourcePoint, timeSignal; iForceField=1, amplitude=source_amplitude)

frames_elastic_full = propagate_linear_frames_with_source(
    preparedElastic,
    Nt;
    sourceFull=sourceFull,
    store_every=store_every,
    blowup_limit=1e30,
)
ux_frames = component_frames(frames_elastic_full, 1)
uz_frames = component_frames(frames_elastic_full, 2)

ux_report = wavefield_snapshot_report(ux_frames)
uz_report = wavefield_snapshot_report(uz_frames)
@show length(frames_elastic_full) ux_report[1] ux_report[end]
@show uz_report[1] uz_report[end]


In [ ]:
fig_ux = plot_wave_snapshots(
    ux_frames;
    sourcePoint=sourcePoint,
    title="2DsismoTimeIsoHetero Marmousi reduced, ux",
)
fig_ux


In [ ]:
fig_uz = plot_wave_snapshots(
    uz_frames;
    sourcePoint=sourcePoint,
    title="2DsismoTimeIsoHetero Marmousi reduced, uz",
)
fig_uz


## 5. Local Stencil Inspection


In [ ]:
st_elastic_eq1_u1 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=1, iField=1)
st_elastic_eq2_u2 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=2, iField=2)

stencil_time_summary(st_elastic_eq1_u1), stencil_time_summary(st_elastic_eq2_u2)
